# Semantic cache

Builds an embedding-based cache (FAISS + `sentence-transformers/all-MiniLM-L6-v2`) so a new prompt can reuse an earlier response if it's close enough in meaning, and measures hit rate / precision-recall / latency on the cleaned test set across several similarity thresholds.

In [1]:
from pathlib import Path
import importlib.util
import sys

ROOT = Path.cwd()
if not (ROOT / 'scripts/evaluate_semantic_cache.py').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'scripts/evaluate_semantic_cache.py').is_file():
    raise RuntimeError('Open this notebook from the repository root or notebooks directory.')

def load(module_name, filename):
    spec = importlib.util.spec_from_file_location(module_name, ROOT / f'scripts/{filename}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

semantic_cache = load('semantic_cache', 'semantic_cache.py')
recover_test_responses = load('recover_test_responses', 'recover_test_responses.py')
evaluate_semantic_cache = load('evaluate_semantic_cache', 'evaluate_semantic_cache.py')
SemanticCache = semantic_cache.SemanticCache

/home/kange/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


## Why an embedding model, not TF-IDF

TF-IDF (already used elsewhere in this repo) measures literal word overlap, so it misses paraphrases that share few words and can even rate unrelated sentences as more similar than real paraphrases.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pairs = [
    ('How do I bake a chocolate cake?', 'What is the recipe for making a chocolate cake?'),
    ('How do I bake a chocolate cake?', 'How do I fix my broken laptop?'),
]
all_texts = [t for pair in pairs for t in pair]
tfidf_vecs = TfidfVectorizer().fit_transform(all_texts)
embedder = SemanticCache(threshold=0.85)

for i, (a, b) in enumerate(pairs):
    t_sim = cosine_similarity(tfidf_vecs[2*i], tfidf_vecs[2*i+1])[0][0]
    cache = SemanticCache(threshold=0.85, model=embedder.model)  # reuse the loaded model, fresh empty cache
    cache.store(a, 'n/a')
    e_sim = cache.lookup(b).similarity
    print(f'{a!r} vs {b!r}')
    print(f'  TF-IDF cosine similarity:    {t_sim:.3f}')
    print(f'  Embedding cosine similarity: {e_sim:.3f}\n')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2140.69it/s]


'How do I bake a chocolate cake?' vs 'What is the recipe for making a chocolate cake?'
  TF-IDF cosine similarity:    0.208
  Embedding cosine similarity: 0.852

'How do I bake a chocolate cake?' vs 'How do I fix my broken laptop?'
  TF-IDF cosine similarity:    0.247
  Embedding cosine similarity: 0.088



/home/kange/AI-Governance-1C-governed-prompt-gateway/scripts/semantic_cache.py:40: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self.model.get_sentence_embedding_dimension()


## Recover response text

`data/test_clean.csv` only has `user_input`/`label`; the cache needs real response text, so this rejoins the raw zip's `model_output` back in. See `docs/semantic-cache.md` for the documented join caveat (24 prompts with ambiguous historical responses).

In [3]:
responses_dir = ROOT / 'data/processed/semantic_cache'
recovered = recover_test_responses.recover(ROOT / 'data/test_clean.csv', ROOT / 'data/lmsys_toxicchat.zip')
responses_dir.mkdir(parents=True, exist_ok=True)
responses_path = responses_dir / 'test_with_responses.csv'
recovered.to_csv(responses_path, index=False)
print(f'Recovered responses for {len(recovered)} rows -> {responses_path}')

Recovered responses for 4883 rows -> /home/kange/AI-Governance-1C-governed-prompt-gateway/data/processed/semantic_cache/test_with_responses.csv


## Evaluate: hit rate, precision/recall against duplicate ground truth, latency

In [4]:
report = evaluate_semantic_cache.evaluate(responses_path, responses_dir)

print(f"Model: {report['model_name']}  |  test rows: {report['test_rows']}")
print(f"Normalized duplicate groups with >1 row: {report['normalized_duplicate_groups_with_more_than_one_row']}\n")
print(f"{'threshold':>9}  {'hit_rate':>9}  {'precision':>9}  {'recall':>9}")
for t in evaluate_semantic_cache.THRESHOLDS:
    s = report['streaming_simulation'][str(t)]
    d = report['duplicate_ground_truth_validation'][str(t)]
    p = f"{d['precision']:.3f}" if d['precision'] is not None else 'n/a'
    r = f"{d['recall']:.3f}" if d['recall'] is not None else 'n/a'
    print(f"{t:>9.2f}  {s['hit_rate']:>9.3f}  {p:>9}  {r:>9}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3384.17it/s]
/home/kange/AI-Governance-1C-governed-prompt-gateway/scripts/semantic_cache.py:40: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self.model.get_sentence_embedding_dimension()


Model: all-MiniLM-L6-v2  |  test rows: 4883
Normalized duplicate groups with >1 row: 115

threshold   hit_rate  precision     recall
     0.70      0.120      0.243      1.000
     0.75      0.097      0.301      1.000
     0.80      0.078      0.358      1.000
     0.85      0.065      0.429      1.000
     0.90      0.053      0.512      1.000
     0.95      0.041      0.644      1.000


In [5]:
lat = report['latency_at_report_threshold']
print(f"Lookup latency at threshold {report['report_threshold']} (real, end-to-end incl. embedding):")
print(f"  mean={lat['mean_seconds']*1000:.2f}ms  median={lat['median_seconds']*1000:.2f}ms  "
      f"p95={lat['p95_seconds']*1000:.2f}ms  max={lat['max_seconds']*1000:.2f}ms")

Lookup latency at threshold 0.85 (real, end-to-end incl. embedding):
  mean=5.59ms  median=4.58ms  p95=12.07ms  max=18.36ms


## Sample hit review

A few of the saved hit sample, for a quick manual sanity check that the reused response actually fits the new prompt

In [6]:
for item in report['hit_sample_at_report_threshold'][:5]:
    print('NEW    :', item['new_prompt'][:100])
    print('MATCHED:', item['matched_prompt'][:100])
    print('SIM    :', round(item['similarity'], 3))
    print('RESPONSE:', item['reused_response'][:100])
    print()

NEW    : what is the weather today?
MATCHED: whats the weather like today
SIM    : 0.852
RESPONSE: I'm sorry, but as an AI language model, I don't have access to real-time weather information. My tra

NEW    : Tell me what you want what you really really want
MATCHED: tell me what you want what you really really want
SIM    : 1.0
RESPONSE: As an AI language model, I don't have personal wants or desires, as I am just a machine. However, I 

NEW    : can you write code?
MATCHED: Can you write code?
SIM    : 1.0
RESPONSE: Yes, I can write code. However, my abilities are limited since I am a language model and not a real 

NEW    : Can you code in python?
MATCHED: can you write python code
SIM    : 0.948
RESPONSE: Yes, I can write Python code. Here is a simple example of a Python program that prints "Hello, World

NEW    : what is your name?
MATCHED: what is your name
SIM    : 0.87
RESPONSE: My name is Assistant. I am an AI trained to assist with a variety of tasks, and I don't have a spec

See [the semantic cache guide](../docs/semantic-cache.md) for the full results table